# Programming Assignment 2: Q Actor-Critic (QAC) for Acrobot

This notebook implements the **Q Actor-Critic (QAC)** algorithm to solve the
`Acrobot-v1` environment (Gymnasium), and reports the performance metrics
needed to evaluate and later compare it against other RL algorithms (e.g. the
tabular Q-learning agent from Assignment 1).

## 1. Problem recap

- Environment: `Acrobot-v1`
- Observation: 6 continuous values `[cos(theta1), sin(theta1), cos(theta2), sin(theta2), thetaDot1, thetaDot2]`
- Actions: 3 discrete torques `{-1, 0, +1}`
- Reward: `-1` per step until the goal height is reached (or truncated at 500 steps)
- Objective: reach the goal in as few steps as possible (i.e., maximize the — less negative — return)

## 2. Why Actor-Critic (and why "Q" Actor-Critic specifically)

Because the observation space is continuous, tabular value storage (as used
for Q-learning) requires discretization, which throws away information and
scales poorly with state dimensionality. Actor-Critic methods instead use
**function approximation** (here, small neural networks) for both:

- **Actor** `π_θ(a|s)` — a parameterized policy (softmax over the 3 discrete
  actions) that is updated using the policy-gradient theorem.
- **Critic** `Q_w(s,a)` — a parameterized action-value function that is
  updated with a TD(0)/SARSA-style bootstrap target and supplies the score
  used by the actor.

This is the classical **QAC** algorithm:

$$
\delta \text{ is not required for the actor update} \qquad Q_w(s,a) \approx Q^{\pi_\theta}(s,a)
$$

$$
\theta \leftarrow \theta + \alpha_\theta \, \nabla_\theta \log \pi_\theta(a|s) \, Q_w(s,a)
$$

$$
w \leftarrow w + \alpha_w \, \bigl[r + \gamma Q_w(s',a') - Q_w(s,a)\bigr] \nabla_w Q_w(s,a)
$$

where `a'` is sampled from the current policy at `s'` (on-policy SARSA-style
bootstrap, since the critic must estimate `Q^{π}`, not `Q*`). Both networks
are updated online, one environment step at a time.


## 3. Performance metrics used to evaluate QAC

To keep results comparable with Assignment 1 (Q-learning) and any future
algorithm, the same *core* training/evaluation metrics are reused, plus a few
metrics specific to actor-critic methods (loss curves, TD error, entropy)
that diagnose *why* the algorithm is (or isn't) learning.

### Core metrics (comparable across algorithms)

1. **Episode return** — sum of rewards per training episode (less negative is better).
2. **Moving-average return** — 50-episode rolling mean, to see the learning trend through the noise.
3. **Episode length** — steps until termination/truncation; shorter usually means a better swing-up policy.
4. **Success rate** — fraction of episodes that terminate (reach the goal) rather than truncate at `max_steps`, evaluated both as a training-time rate and via periodic greedy checkpoints.
5. **Average steps-to-goal** — mean episode length computed only over *successful* evaluation episodes.
6. **Training wall-clock time** — total seconds to run all training episodes.
7. **Sample efficiency** — first episode index where the 50-episode moving-average return crosses a target threshold.
8. **Final evaluation mean/std reward, success rate, mean episode length** — measured with a purely greedy (`argmax π`) policy, separate from the exploratory training policy.

### Actor-Critic-specific diagnostic metrics

9. **Critic loss (TD-error²)** — mean squared TD error per episode; should trend down and stabilize as `Q_w` converges.
10. **Mean |TD error|** — average magnitude of the bootstrap error; large/non-decreasing values indicate critic instability or too-high a learning rate.
11. **Actor loss** — mean `-log π(a|s)·Q_w(s,a)` per episode; used to check the policy-gradient signal is well-scaled (not exploding/vanishing).
12. **Policy entropy** — mean entropy of `π(·|s)` per episode; should start high (near `log 3 ≈ 1.10`, uniform over 3 actions) and gradually decrease as the policy commits to better actions, without collapsing too early (which would indicate premature convergence / insufficient exploration).

All of these are logged per-episode during training, written to CSV files,
and plotted below.


In [ ]:
"""
Install (if needed):
    pip install gymnasium[classic-control] numpy matplotlib pandas torch
"""
import os
import time

import gymnasium as gym
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.distributions import Categorical

DEVICE = torch.device("cpu")
print(f"PyTorch version : {torch.__version__}")
print(f"Gymnasium version: {gym.__version__}")
print(f"Using device      : {DEVICE}")


In [ ]:
# ---------------------------------------------------------------------------
# Environment setup & inspection
# ---------------------------------------------------------------------------
env = gym.make("Acrobot-v1")
eval_env = gym.make("Acrobot-v1")

STATE_DIM = env.observation_space.shape[0]
ACTION_DIM = env.action_space.n

print(f"Observation space : {env.observation_space}  (dim={STATE_DIM})")
print(f"Action space      : {env.action_space}  (n={ACTION_DIM})")


In [ ]:
# ---------------------------------------------------------------------------
# CONFIG — all hyperparameters in one place
# ---------------------------------------------------------------------------
CONFIG = {
    "seed": 42,
    "hidden_size": 64,
    "actor_lr": 5e-4,
    "critic_lr": 1e-3,
    "gamma": 0.99,
    "entropy_coef": 0.01,
    "num_episodes": 1000,
    "max_steps": 500,
    "eval_interval": 100,     # evaluate every N training episodes
    "eval_episodes": 10,      # episodes per checkpoint evaluation
    "final_eval_episodes": 100,
    "ma_window": 50,          # moving-average window
    # Sample-efficiency threshold: first episode where 50-ep MA exceeds this
    "efficiency_threshold": -300.0,
}

SEED = CONFIG["seed"]
np.random.seed(SEED)
torch.manual_seed(SEED)
env.reset(seed=SEED)
env.action_space.seed(SEED)
eval_env.reset(seed=SEED + 1)

print("Config:")
for k, v in CONFIG.items():
    print(f"  {k:<22} {v}")


In [ ]:
# ---------------------------------------------------------------------------
# Actor (policy) & Critic (action-value) networks
# ---------------------------------------------------------------------------

class Actor(nn.Module):
    """pi_theta(a|s): outputs a categorical distribution over discrete actions."""

    def __init__(self, state_dim: int, action_dim: int, hidden_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, action_dim),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        logits = self.net(state)
        return torch.softmax(logits, dim=-1)


class Critic(nn.Module):
    """Q_w(s, .): outputs one Q-value per discrete action given a state."""

    def __init__(self, state_dim: int, action_dim: int, hidden_size: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(state_dim, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, hidden_size),
            nn.Tanh(),
            nn.Linear(hidden_size, action_dim),
        )

    def forward(self, state: torch.Tensor) -> torch.Tensor:
        return self.net(state)


def make_networks(cfg: dict):
    actor = Actor(STATE_DIM, ACTION_DIM, cfg["hidden_size"]).to(DEVICE)
    critic = Critic(STATE_DIM, ACTION_DIM, cfg["hidden_size"]).to(DEVICE)
    actor_opt = optim.Adam(actor.parameters(), lr=cfg["actor_lr"])
    critic_opt = optim.Adam(critic.parameters(), lr=cfg["critic_lr"])
    return actor, critic, actor_opt, critic_opt


In [ ]:
# ---------------------------------------------------------------------------
# Single-episode rollout — shared by training (online QAC updates) and
# greedy evaluation (no updates, argmax action selection).
# ---------------------------------------------------------------------------

def run_episode(env: gym.Env, actor: Actor, critic: Critic,
                 actor_opt: optim.Optimizer, critic_opt: optim.Optimizer,
                 cfg: dict, greedy: bool = False) -> dict:
    """
    Roll out one episode.

    If greedy=False: performs online QAC updates (actor + critic) after
    every environment step, using an on-policy SARSA-style bootstrap target
    for the critic:  delta = r + gamma * Q_w(s', a') - Q_w(s, a)
    and the policy-gradient actor update: theta += lr * grad(log pi(a|s)) * Q_w(s,a).

    If greedy=True: no learning happens; actions are chosen as argmax pi(.|s).

    Returns a dict with: reward, length, success, critic_loss, actor_loss,
    td_error, entropy (the last four are episode means, NaN when greedy=True).
    """
    gamma = cfg["gamma"]
    entropy_coef = cfg["entropy_coef"]
    max_steps = cfg["max_steps"]

    state, _ = env.reset()
    state_t = torch.as_tensor(state, dtype=torch.float32, device=DEVICE)

    total_reward = 0.0
    success = 0
    critic_losses, actor_losses, td_errors, entropies = [], [], [], []

    for step in range(max_steps):
        probs = actor(state_t)
        dist = Categorical(probs)

        if greedy:
            action = int(torch.argmax(probs).item())
        else:
            action_t = dist.sample()
            action = int(action_t.item())
            log_prob = dist.log_prob(action_t)
            entropy = dist.entropy()

        next_state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
        next_state_t = torch.as_tensor(next_state, dtype=torch.float32, device=DEVICE)

        if not greedy:
            with torch.no_grad():
                if done:
                    target = torch.tensor(float(reward), device=DEVICE)
                else:
                    next_probs = actor(next_state_t)
                    next_action = int(Categorical(next_probs).sample().item())
                    next_q_values = critic(next_state_t)
                    target = reward + gamma * next_q_values[next_action]

            # --- Critic update: minimize TD error^2 ---
            q_values = critic(state_t)
            q_sa = q_values[action]
            td_error = target - q_sa
            critic_loss = td_error.pow(2)

            critic_opt.zero_grad()
            critic_loss.backward()
            critic_opt.step()

            # --- Actor update: policy gradient scored by Q_w(s,a) ---
            actor_loss = -(log_prob * q_sa.detach()) - entropy_coef * entropy

            actor_opt.zero_grad()
            actor_loss.backward()
            actor_opt.step()

            critic_losses.append(critic_loss.item())
            actor_losses.append(actor_loss.item())
            td_errors.append(abs(td_error.item()))
            entropies.append(entropy.item())

        total_reward += reward
        if terminated:
            success = 1
        state_t = next_state_t

        if done:
            break

    return {
        "reward": total_reward,
        "length": step + 1,
        "success": success,
        "critic_loss": float(np.mean(critic_losses)) if critic_losses else float("nan"),
        "actor_loss": float(np.mean(actor_losses)) if actor_losses else float("nan"),
        "td_error": float(np.mean(td_errors)) if td_errors else float("nan"),
        "entropy": float(np.mean(entropies)) if entropies else float("nan"),
    }


In [ ]:
# ---------------------------------------------------------------------------
# Greedy evaluation — runs `episodes` greedy rollouts, no learning.
# ---------------------------------------------------------------------------

def evaluate(env: gym.Env, actor: Actor, critic: Critic, cfg: dict, episodes: int) -> dict:
    rewards, lengths, successes = [], [], []
    for _ in range(episodes):
        result = run_episode(env, actor, critic, None, None, cfg, greedy=True)
        rewards.append(result["reward"])
        lengths.append(result["length"])
        successes.append(result["success"])
    return {"rewards": rewards, "lengths": lengths, "successes": successes}


# ---------------------------------------------------------------------------
# Training loop
# ---------------------------------------------------------------------------

def train(env: gym.Env, eval_env: gym.Env, actor: Actor, critic: Critic,
          actor_opt: optim.Optimizer, critic_opt: optim.Optimizer, cfg: dict) -> dict:
    num_episodes = cfg["num_episodes"]
    eval_interval = cfg["eval_interval"]
    ma_window = cfg["ma_window"]

    history = {
        "rewards": [], "lengths": [], "successes": [],
        "critic_losses": [], "actor_losses": [], "td_errors": [], "entropies": [],
        "checkpoint_episodes": [], "checkpoint_success_rates": [],
        "checkpoint_mean_rewards": [], "checkpoint_mean_lengths": [],
    }

    t_start = time.time()

    for episode in range(1, num_episodes + 1):
        result = run_episode(env, actor, critic, actor_opt, critic_opt, cfg, greedy=False)

        history["rewards"].append(result["reward"])
        history["lengths"].append(result["length"])
        history["successes"].append(result["success"])
        history["critic_losses"].append(result["critic_loss"])
        history["actor_losses"].append(result["actor_loss"])
        history["td_errors"].append(result["td_error"])
        history["entropies"].append(result["entropy"])

        if episode % eval_interval == 0:
            ckpt = evaluate(eval_env, actor, critic, cfg, cfg["eval_episodes"])
            success_rate = 100.0 * float(np.mean(ckpt["successes"]))
            history["checkpoint_episodes"].append(episode)
            history["checkpoint_success_rates"].append(success_rate)
            history["checkpoint_mean_rewards"].append(float(np.mean(ckpt["rewards"])))
            history["checkpoint_mean_lengths"].append(float(np.mean(ckpt["lengths"])))

            recent = history["rewards"][-ma_window:]
            ma = np.mean(recent)
            print(
                f"Episode {episode:5d}/{num_episodes} | "
                f"MA Reward({ma_window}): {ma:8.1f} | "
                f"Eval success: {success_rate:5.1f}% | "
                f"Entropy: {result['entropy']:.3f} | "
                f"CriticLoss: {result['critic_loss']:.2f}"
            )

    history["training_time"] = time.time() - t_start
    return history


In [ ]:
# ---------------------------------------------------------------------------
# Metrics & CSV output
# ---------------------------------------------------------------------------

def compute_metrics(history: dict, final_eval: dict, cfg: dict) -> dict:
    rewards = np.array(history["rewards"])
    w = cfg["ma_window"]

    ma_rewards = np.convolve(rewards, np.ones(w) / w, mode="valid")
    best_ma = float(np.max(ma_rewards)) if len(ma_rewards) > 0 else float("nan")

    threshold = cfg["efficiency_threshold"]
    crossing = next(
        (i + w for i, v in enumerate(ma_rewards) if v >= threshold),
        None
    )

    eval_rewards = np.array(final_eval["rewards"])
    eval_lengths = np.array(final_eval["lengths"])
    eval_successes = np.array(final_eval["successes"])
    success_mask = eval_successes.astype(bool)
    avg_steps_to_goal = (
        float(np.mean(eval_lengths[success_mask])) if success_mask.any() else float("nan")
    )

    tail = min(100, len(history["rewards"]))

    return {
        "total_episodes":        cfg["num_episodes"],
        "best_ma_reward":        round(best_ma, 2),
        "eval_mean_reward":      round(float(np.mean(eval_rewards)), 2),
        "eval_std_reward":       round(float(np.std(eval_rewards)), 2),
        "eval_success_rate_pct": round(100.0 * float(np.mean(eval_successes)), 1),
        "eval_mean_ep_length":   round(float(np.mean(eval_lengths)), 1),
        "avg_steps_to_goal":     round(avg_steps_to_goal, 1),
        "training_time_sec":     round(history["training_time"], 1),
        "sample_efficiency_ep":  crossing,
        "final_critic_loss":     round(float(np.nanmean(history["critic_losses"][-tail:])), 4),
        "final_actor_loss":      round(float(np.nanmean(history["actor_losses"][-tail:])), 4),
        "final_mean_abs_td_error": round(float(np.nanmean(history["td_errors"][-tail:])), 4),
        "final_entropy":         round(float(np.nanmean(history["entropies"][-tail:])), 4),
    }


def save_csv(history: dict, path: str) -> None:
    df = pd.DataFrame({
        "episode":     range(1, len(history["rewards"]) + 1),
        "reward":      history["rewards"],
        "length":      history["lengths"],
        "success":     history["successes"],
        "critic_loss": history["critic_losses"],
        "actor_loss":  history["actor_losses"],
        "td_error":    history["td_errors"],
        "entropy":     history["entropies"],
    })
    df.to_csv(path, index=False)
    print(f"Saved training history -> {path}")


def save_summary_csv(metrics: dict, path: str) -> None:
    pd.DataFrame([metrics]).to_csv(path, index=False)
    print(f"Saved summary metrics -> {path}")


def save_checkpoint_csv(history: dict, path: str) -> None:
    df = pd.DataFrame({
        "episode":            history["checkpoint_episodes"],
        "success_rate_pct":   history["checkpoint_success_rates"],
        "mean_eval_reward":   history["checkpoint_mean_rewards"],
        "mean_eval_length":   history["checkpoint_mean_lengths"],
    })
    df.to_csv(path, index=False)
    print(f"Saved checkpoint success rates -> {path}")


def print_summary(metrics: dict) -> None:
    print("\n" + "=" * 56)
    print("  SUMMARY TABLE — Q Actor-Critic (Acrobot-v1)")
    print("=" * 56)
    rows = [
        ("Total training episodes",       metrics["total_episodes"]),
        ("Best moving-avg reward",        metrics["best_ma_reward"]),
        ("Eval mean reward",              metrics["eval_mean_reward"]),
        ("Eval std reward",               metrics["eval_std_reward"]),
        ("Eval success rate (%)",         metrics["eval_success_rate_pct"]),
        ("Eval mean episode length",      metrics["eval_mean_ep_length"]),
        ("Avg steps-to-goal (success)",   metrics["avg_steps_to_goal"]),
        ("Training time (s)",             metrics["training_time_sec"]),
        ("Sample efficiency (episode)",
         metrics["sample_efficiency_ep"] if metrics["sample_efficiency_ep"] else "Not reached"),
        ("Final critic loss (MSE TD)",    metrics["final_critic_loss"]),
        ("Final actor loss",              metrics["final_actor_loss"]),
        ("Final mean |TD error|",         metrics["final_mean_abs_td_error"]),
        ("Final policy entropy",          metrics["final_entropy"]),
    ]
    for label, value in rows:
        print(f"  {label:<32} {value}")
    print("=" * 56 + "\n")


In [ ]:
# ---------------------------------------------------------------------------
# Visualizations
# ---------------------------------------------------------------------------

def _moving_average(data: list, window: int) -> np.ndarray:
    return np.convolve(data, np.ones(window) / window, mode="valid")


def _plot_series(y, cfg, out_dir, filename, title, ylabel, color, ma_color=None, hline=None):
    ma = _moving_average(y, cfg["ma_window"])
    x = range(1, len(y) + 1)
    x_ma = range(cfg["ma_window"], len(y) + 1)

    plt.figure(figsize=(10, 4))
    plt.plot(x, y, alpha=0.25, color=color, label=title)
    plt.plot(x_ma, ma, color=ma_color or color, linewidth=2,
              label=f"{cfg['ma_window']}-ep moving average")
    if hline is not None:
        plt.axhline(hline, color="red", linestyle="--", linewidth=0.8, label="Worst possible")
    plt.xlabel("Episode")
    plt.ylabel(ylabel)
    plt.title(f"QAC — {title} (Acrobot-v1)")
    plt.legend()
    plt.tight_layout()
    path = os.path.join(out_dir, filename)
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved plot -> {path}")


def plot_reward_curve(history, cfg, out_dir):
    _plot_series(history["rewards"], cfg, out_dir, "reward_curve.png",
                 "Episode Reward", "Total reward", "steelblue", hline=-500)


def plot_length_curve(history, cfg, out_dir):
    _plot_series(history["lengths"], cfg, out_dir, "episode_length_curve.png",
                 "Episode Length", "Steps", "darkorange")


def plot_critic_loss_curve(history, cfg, out_dir):
    _plot_series(history["critic_losses"], cfg, out_dir, "critic_loss_curve.png",
                 "Critic Loss (TD error^2)", "MSE", "firebrick")


def plot_actor_loss_curve(history, cfg, out_dir):
    _plot_series(history["actor_losses"], cfg, out_dir, "actor_loss_curve.png",
                 "Actor Loss", "-log(pi)*Q - entropy bonus", "teal")


def plot_entropy_curve(history, cfg, out_dir):
    _plot_series(history["entropies"], cfg, out_dir, "entropy_curve.png",
                 "Policy Entropy", "Entropy (nats)", "mediumpurple")


def plot_success_rate(history, out_dir):
    ckpt_eps = history["checkpoint_episodes"]
    ckpt_rates = history["checkpoint_success_rates"]

    plt.figure(figsize=(8, 4))
    plt.plot(ckpt_eps, ckpt_rates, marker="o", color="seagreen", linewidth=2)
    plt.xlabel("Training episode")
    plt.ylabel("Success rate (%)")
    plt.title("QAC — Checkpoint Success Rate (Acrobot-v1)")
    plt.ylim(0, 105)
    plt.tight_layout()
    path = os.path.join(out_dir, "success_rate.png")
    plt.savefig(path, dpi=150)
    plt.close()
    print(f"Saved plot -> {path}")


In [ ]:
# ---------------------------------------------------------------------------
# Run training end-to-end
# ---------------------------------------------------------------------------

OUT_DIR = os.path.dirname(os.path.abspath("Actor-CriticAlgorithm.ipynb")) or "."
FIG_DIR = os.path.join(OUT_DIR, "figures")
os.makedirs(FIG_DIR, exist_ok=True)

actor, critic, actor_opt, critic_opt = make_networks(CONFIG)

history = train(env, eval_env, actor, critic, actor_opt, critic_opt, CONFIG)

print("\nRunning final greedy evaluation...")
final_eval = evaluate(eval_env, actor, critic, CONFIG, CONFIG["final_eval_episodes"])

metrics = compute_metrics(history, final_eval, CONFIG)
print_summary(metrics)

save_csv(history, os.path.join(OUT_DIR, "results_actor_critic.csv"))
save_summary_csv(metrics, os.path.join(OUT_DIR, "summary_metrics_actor_critic.csv"))
save_checkpoint_csv(history, os.path.join(OUT_DIR, "checkpoint_success_actor_critic.csv"))

plot_reward_curve(history, CONFIG, FIG_DIR)
plot_length_curve(history, CONFIG, FIG_DIR)
plot_success_rate(history, FIG_DIR)
plot_critic_loss_curve(history, CONFIG, FIG_DIR)
plot_actor_loss_curve(history, CONFIG, FIG_DIR)
plot_entropy_curve(history, CONFIG, FIG_DIR)

env.close()
eval_env.close()
